# V3 EDA 3 — 단지 단위 1년 가격 변화: 정답으로 쓸 수 있나, 예측할 단서가 있나

## 목적

31(구 × 월)은 25개 구라 표본이 적고, 3개월 뒤 구 순위는 최근 4주 흐름 이상으로 예측되지 않았다. 설계서 Phase 4-3은 단위를 단지로 바꿔
"향후 1년 같은 구 평균보다 더 오를 단지"(상대 정답)를 설계할지 정하기로 했다. 그 전에 두 가지를 확인한다.

## 통과 기준 (결과를 보기 전에 고정, 2026-11-12)

평가 구간: 2023→2024, 2024→2025 (2025→2026은 Test 기간이 들어가므로 쓰지 않는다). **둘 다 통과할 때만** 단지 단위 분류를 설계한다.

1. **정답 신뢰도:** 같은 단지의 거래를 무작위로 반씩 나눠 각각 구한 상대 변화율의 상관(시드 20개 평균)을 Spearman-Brown으로 전체 표본 기준으로 환산한 값이 **두 구간 모두 ≥ 0.5**
2. **예측 단서:** 아래 단순 신호 중 하나라도 다음 해 상대 변화율과의 순위 상관이 **두 구간 모두 ≥ 0.1**
   - 직전 1년 흐름 (모멘텀)
   - 적정가 대비 싸게 거래된 정도 (11 OOF 잔차 평균, 음수 = 싸게 거래)
   - 신호와 정답이 같은 거래를 쓰면 평균 회귀 때문에 상관이 저절로 생긴다 → 기준 연도 거래를 반으로 나눠 **신호는 A 절반, 정답의 출발점은 B 절반**으로 계산한다

## 정의

- 대상: 중개거래, 같은 단지 · 면적(전용 1㎡ 반올림) 묶음
- 변화율 = 다음 해 log 가격 중앙값 − 기준 연도 log 가격 중앙값 (같은 면적끼리), 단지 값 = 면적 묶음 평균
- 상대 변화율 = 단지 변화율 − 같은 구 단지들의 중앙값
- 표본 조건: 각 연도 · 각 절반에 같은 면적 거래 2건 이상

## 결론 요약

| 기준 (결과 전 고정) | 2023→24 | 2024→25 | 판정 |
|---|---:|---:|---|
| 1. 정답 신뢰도 (S-B) ≥ 0.5 | 0.60 (1,246단지) | 0.69 (1,832단지) | 통과 |
| 2-a. 모멘텀 순위 상관 ≥ 0.1 | 0.03 | 0.28 | 미달 (국면에 따라 다름) |
| 2-b. 저평가(−r) 순위 상관 ≥ 0.1 | −0.24 | −0.22 | 미달 (방향 반대) |

- **판정: 단지 단위 분류는 설계하지 않는다** (31 지역 흐름 지표 유지).
- 정답 자체는 쓸 만하다: 단지 상대 1년 변화율은 반씩 나눠도 서로 맞는다 (신뢰도 0.60~0.69). 상대 변화율 퍼짐은 10~90% 구간 약 −6% ~ +6~8%.
- **관찰 (채택 아님, 결과를 본 뒤 알게 된 방향):** 저평가 신호는 두 구간 모두 **반대 방향으로 일관**됐다. 적정가보다 비싸게 거래되던(OOF 잔차 +) 단지가 다음 해에도 같은 구 평균보다 더 올랐다.
  11 적정가가 오르는 단지를 늦게 따라간다는 뜻으로 읽힌다. 31의 기회 신호(적정가보다 5% 넘게 싼 **개별 거래**는 다음 거래에서 더 오름)는 거래 단위 잡음의 되돌림이고, 이것은 단지 단위 흐름의 지속이라 서로 모순되지 않는다.
  이 방향을 쓰려면 새 가설로 다시 고정하고 아직 보지 않은 기간에서 확인해야 한다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

pd.set_option("display.width", 200)


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / ".env").exists():
            return path
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
trade = pd.read_parquet(PROJECT_DIR / "data/v2_5year_seoul/processed/v2_trade.parquet",
                        columns=["transaction_id", "단지일련번호", "시군구명", "계약일자", "전용면적", "거래유형", "거래금액"])
oof = pd.read_parquet(PROJECT_DIR / "models/regression/v2/oof_predictions.parquet", columns=["transaction_id", "r"])
t = trade[trade["거래유형"].eq("중개거래")].merge(oof, on="transaction_id", how="left", validate="one_to_one")
t["연"] = pd.to_datetime(t["계약일자"]).dt.year
t = t[t["연"].between(2022, 2025)].copy()          # 2026(Test)은 쓰지 않는다
t["단지"] = t["단지일련번호"].astype(str)
t["면적키"] = t["전용면적"].astype(float).round(0)
t["logp"] = np.log(t["거래금액"].astype(float))
print(t.groupby("연").agg(거래=("logp", "size"), 단지=("단지", "nunique"), OOF있음=("r", lambda s: s.notna().mean())).round(3))
PAIRS = [(2023, 2024), (2024, 2025)]
SEEDS = range(20)

         거래    단지  OOF있음
연                       
2022  10091  3411    1.0
2023  31734  4274    1.0
2024  52741  5005    1.0
2025  74048  5530    1.0


## 1. 정답 신뢰도 — 반씩 나눈 두 추정치가 서로 맞나

In [2]:
def assign_half(df, seed):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    x = df.iloc[order].copy()
    x["half"] = x.groupby(["단지", "면적키", "연"]).cumcount() % 2   # 묶음 안에서 번갈아 → 절반 크기 균형
    return x


def med(df, mask):
    g = df[mask].groupby(["단지", "면적키"])["logp"]
    return g.median().where(g.size() >= 2)


def to_complex_relative(area_delta, gu):
    s = area_delta.dropna().groupby(level="단지").mean()
    rel = s - s.groupby(gu.reindex(s.index)).transform("median")
    return rel


gu = t.drop_duplicates("단지").set_index("단지")["시군구명"]
rows = []
for y0, y1 in [(2022, 2023)] + PAIRS:
    cors, ns = [], []
    for seed in SEEDS:
        x = assign_half(t, seed)
        dA = med(x, (x["연"] == y1) & (x["half"] == 0)) - med(x, (x["연"] == y0) & (x["half"] == 0))
        dB = med(x, (x["연"] == y1) & (x["half"] == 1)) - med(x, (x["연"] == y0) & (x["half"] == 1))
        both = dA.dropna().index.intersection(dB.dropna().index)
        a, b = to_complex_relative(dA.loc[both], gu), to_complex_relative(dB.loc[both], gu)
        cors.append(spearmanr(a, b.reindex(a.index)).correlation)
        ns.append(len(a))
    r_half = float(np.mean(cors))
    rows.append({"구간": f"{y0}→{y1}", "단지 수": int(np.mean(ns)), "절반끼리 상관": r_half, "전체 기준 신뢰도(S-B)": 2 * r_half / (1 + r_half),
                 "기준(≥0.5)": "평가 구간 아님" if (y0, y1) not in PAIRS else ("통과" if 2 * r_half / (1 + r_half) >= 0.5 else "미달")})
reliability = pd.DataFrame(rows)
reliability.round(3)

,구간,단지 수,절반끼리 상관,전체 기준 신뢰도(S-B),기준(≥0.5)
0,2022→2023,358,0.488,0.656,평가 구간 아님
1,2023→2024,1246,0.429,0.601,통과
2,2024→2025,1832,0.521,0.685,통과


## 2. 예측 단서 — 기준 연도 A 절반의 신호 vs B 절반에서 출발한 다음 해 상대 변화율

- 정답(B): 다음 해 전체 거래 중앙값 − 기준 연도 B 절반 중앙값 → 상대화
- 모멘텀(A): 기준 연도 A 절반 중앙값 − 전년 전체 중앙값 → 상대화
- 저평가(A): 기준 연도 A 절반 거래의 11 OOF 잔차 평균 (`r = log(실거래/적정가)`, 음수 = 적정가보다 싸게 거래) → 상대화. 다음 해에 더 오를 것이라는 가설이면 **음의 상관**이 단서다 → 부호를 뒤집어(−r) 비교한다

In [3]:
rows = []
for y0, y1 in PAIRS:
    res = {"모멘텀": [], "저평가(−r)": []}
    ns = {"모멘텀": [], "저평가(−r)": []}
    for seed in SEEDS:
        x = assign_half(t, seed)
        g_next = x[x["연"] == y1].groupby(["단지", "면적키"])["logp"]
        next_all = g_next.median().where(g_next.size() >= 2)
        g_prev = x[x["연"] == y0 - 1].groupby(["단지", "면적키"])["logp"]
        prev_all = g_prev.median().where(g_prev.size() >= 2)
        baseB = med(x, (x["연"] == y0) & (x["half"] == 1))
        baseA = med(x, (x["연"] == y0) & (x["half"] == 0))
        label = to_complex_relative(next_all - baseB, gu)
        mom = to_complex_relative(baseA - prev_all, gu)
        ra = x[(x["연"] == y0) & (x["half"] == 0)].groupby("단지")["r"].mean()
        ra = -(ra - ra.groupby(gu.reindex(ra.index)).transform("median"))
        for name, sig in [("모멘텀", mom), ("저평가(−r)", ra)]:
            idx = label.index.intersection(sig.dropna().index)
            res[name].append(spearmanr(label.loc[idx], sig.loc[idx]).correlation)
            ns[name].append(len(idx))
    for name in res:
        rows.append({"구간": f"{y0}→{y1}", "신호": name, "단지 수": int(np.mean(ns[name])), "순위 상관": float(np.mean(res[name])),
                     "시드 범위": f"{np.min(res[name]):.3f} ~ {np.max(res[name]):.3f}"})
signals = pd.DataFrame(rows)
signals.round(3)

,구간,신호,단지 수,순위 상관,시드 범위
0,2023→2024,모멘텀,759,0.031,0.000 ~ 0.058
1,2023→2024,저평가(−r),1343,-0.240,-0.270 ~ -0.205
2,2024→2025,모멘텀,1685,0.284,0.260 ~ 0.308
3,2024→2025,저평가(−r),2012,-0.222,-0.243 ~ -0.193


## 3. 판정

In [4]:
ok1 = reliability[reliability["구간"].isin([f"{a}→{b}" for a, b in PAIRS])]["전체 기준 신뢰도(S-B)"].ge(0.5).all()
best = signals.pivot(index="신호", columns="구간", values="순위 상관")
ok2_by_signal = best.ge(0.1).all(axis=1)
print("1. 정답 신뢰도 ≥ 0.5 (두 구간):", "통과" if ok1 else "미달")
print("2. 예측 단서 ≥ 0.1 (두 구간):", ok2_by_signal.to_dict(), "→", "통과" if ok2_by_signal.any() else "미달")
print("→ 단지 단위 분류:", "설계 진행" if ok1 and ok2_by_signal.any() else "설계하지 않음 (31 지역 흐름 지표 유지)")
best.round(3)

1. 정답 신뢰도 ≥ 0.5 (두 구간): 통과
2. 예측 단서 ≥ 0.1 (두 구간): {'모멘텀': False, '저평가(−r)': False} → 미달
→ 단지 단위 분류: 설계하지 않음 (31 지역 흐름 지표 유지)


구간,2023→2024,2024→2025
신호,,
모멘텀,0.031,0.284
저평가(−r),-0.240,-0.222


## 4. 참고 — 상대 변화율 크기

상대 변화율의 퍼짐이 얼마나 되는지 (정답으로 쓸 때 "의미 있는 차이"의 크기 감).

In [5]:
x = t.copy()
rows = []
for y0, y1 in PAIRS:
    g0 = x[x["연"] == y0].groupby(["단지", "면적키"])["logp"]
    g1 = x[x["연"] == y1].groupby(["단지", "면적키"])["logp"]
    d = (g1.median().where(g1.size() >= 3) - g0.median().where(g0.size() >= 3))
    rel = to_complex_relative(d, gu)
    q = rel.quantile([.1, .25, .5, .75, .9])
    rows.append({"구간": f"{y0}→{y1}", "단지 수": len(rel), **{f"{int(k * 100)}%": (np.exp(v) - 1) * 100 for k, v in q.items()}})
pd.DataFrame(rows).round(2)

,구간,단지 수,10%,25%,50%,75%,90%
0,2023→2024,1567,-5.77,-2.89,0.0,3.09,6.04
1,2024→2025,2210,-6.40,-3.12,0.0,3.49,7.55


## 5. 정리

| 항목 | 결과 | 다음 |
|---|---|---|
| 정답 신뢰도 | 통과 (0.60 · 0.69) | — |
| 예측 단서 | 모멘텀은 2023→24 0.03 · 2024→25 0.28로 국면 의존, 저평가는 −0.24 · −0.22로 가설과 반대 | 단지 단위 분류 설계 안 함 |
| 관찰 | 적정가보다 비싸게 거래되는 단지가 계속 더 오름 (두 구간 일관) | 회귀 쪽 후보로 쓸지(단지 최근 OOF 잔차 Feature, 시점 규칙 지킴)는 V3 범위 밖이라 사용자 결정 |

V3 EDA 목록(1 · 2 · 3)은 여기서 끝난다. EDA 1 · 2에서 회귀 시험을 통과할 후보가 나오지 않았고, EDA 3은 분류 설계 조건에 미달했다.